# מעבדה 17 — סטטיסטיקה קוונטית: גזי בוזה–איינשטיין ופרמי–דיראק

במעבדה הזו אתם מוסיפים לקורס כלל אחד — כמה חלקיקים זהים רשאים לחלוק אותו מצב חד-חלקיקי — ועוקבים
אחר מה שהוא עושה. אתם סופרים את המצבים של שני חלקיקים תחת שלושה כללים; בונים מחדש את האכלוסים של
בוזה–איינשטיין ושל פרמי–דיראק מסכומים מפורשים ומגרילים אותם באקראי; סופרים את המצבים של קופסה אחד
אחד; פותרים עבור הפוטנציאל הכימי ששומר על גז בצפיפות קבועה, וצופים בו מטפס כשהגז מתקרר; עוקבים אחר
גז בוזה לאורך העיבוי, הן בגבול האידיאלי והן בקופסה ממשית של עד מיליון אטומים; ממלאים ים פרמי ומודדים
את קיבול החום שלו; בונים מחדש את ספקטרום פלאנק של מודול 16 מאכלוס של בוזונים; ולבסוף מודדים את
אלקטרוני ההולכה של כסף בנתונים ממשיים, ובודקים את גז בוזה האידיאלי מול הליום נוזלי.

המסלול: האובייקטים (חלק 1); ספירה (חלק 2); מצב אחד והפלוקטואציות שלו (חלק 3); המצבים של קופסה
(חלק 4); הפוטנציאל הכימי בצפיפות קבועה (חלק 5); עיבוי בוזה–איינשטיין (חלק 6); ים פרמי (חלק 7); חוק
פלאנק כגז בוזה (חלק 8); האלקטרונים בכסף (חלק 9); הליום נוזלי (חלק 10); הבדיקות האוטומטיות (חלק 11);
וארבעה כלי חקירה (חלק 12).

עברו עליה בסדר. במקום שבו המחברת מבקשת מכם לנבא, רשמו את ניבויכם בתא המיועד לכך **לפני** הרצת התא
הבא. ניבוי שהתחייבתם אליו הוא הדרך האמינה היחידה לגלות שטעיתם.

**יחידות.** יחידות SI לכל אורך הדרך: אנרגיות בג'אול (אלקטרון-וולט רק בערכים המודפסים), צפיפויות
ב-$\mathrm{m^{-3}}$, טמפרטורות בקלווין, קיבולי חום ב-$\mathrm{J\,K^{-1}}$ — למול, כאשר מספר החלקיקים
הוא $N_A$.

## מפרט המודל

| | |
|---|---|
| **מערכת** | גז אידיאלי של חלקיקים זהים — בוזונים, פרמיונים או חלקיקים קלאסיים — בקופסה קובייתית קשיחה בנפח $V$, וכל גל עומד בעל ניוון ספין $g_s$ |
| **דינמיקה** | אין — כל עקומה היא שיווי משקל תרמי; המספרים האקראיים היחידים מוגרלים במכוון, דרך מחולל המועבר כארגומנט |
| **גבול** | כל מצב חד-חלקיקי מחליף אנרגיה וחלקיקים עם מאגר ב-$(T, \mu)$; פותרים עבור $\mu$ כך שהגז יכיל $N$ חלקיקים |
| **צבר** | גרנד-קנוני, מצב אחר מצב |
| **מוזנח** | כל אינטראקציה בין החלקיקים; הצורה של מלכודת ממשית; יחסות; עבור אלקטרונים, הסריג מעבר לקופסת אלקטרונים חופשיים |
| **תקף כאשר** | המרווח בין הרמות בקופסה קטן בהרבה מ-$\kB T$ (ומ-$\varepsilon_F$); $N \gg 1$ עבור עיבוי |
| **אופני כישלון** | הליום-4 נוזלי (אינטראקציות חזקות); מתכות ממשיות (מבנה פסים); כמה מאות אטומים (מעבר מעוגל); ננסים לבנים סמוך למסת צ'נדראסקאר (אלקטרונים יחסותיים) |

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from thermolab import chemical, quantum, radiation
from thermolab.constants import K_B, N_A
from thermolab.partition import thermal_wavelength
from thermolab.validation import relative_error, scaling_exponent

H = radiation.PLANCK_H
EV = 1.602176634e-19  # J

try:
    import piplite  # noqa: F401
except ImportError:
    # Desktop / nbmake: the repository root is three directories up from this notebook.
    DATA = Path("..", "..", "..", "data")
else:
    # JupyterLite bundles only the notebooks/ tree (jupyter_lite_config.json's
    # LiteBuildConfig.contents), so the browser gets its own copy of the CSVs co-located here.
    DATA = Path("data")

# The two gases this laboratory keeps returning to.
M_HE, N_HE = quantum.M_HELIUM4, 146.1 / quantum.M_HELIUM4   # liquid helium-4 at its lambda point
M_E, N_CU = quantum.M_ELECTRON, 8960.0 / 63.546e-3 * N_A    # copper's conduction electrons
T_C = quantum.bec_tc(N_HE, M_HE)
T_F = float(quantum.fermi_temperature(N_CU, M_E))
print(f"helium-4 at 146.1 kg/m^3: n = {N_HE:.4e} m^-3, ideal-gas T_c = {T_C:.3f} K")
print(f"copper's electrons: n = {N_CU:.4e} m^-3, "
      f"eps_F = {quantum.fermi_energy(N_CU, M_E) / EV:.3f} eV, T_F = {T_F:.0f} K")

## חלק 1 — האובייקטים

| פונקציה | מה היא נותנת |
|---|---|
| `count_states(M, N, s)`, `enumerate_states(M, N, s)` | המספר, והרשימה, של המצבים של $N$ חלקיקים ב-$M$ מצבים חד-חלקיקיים, עבור `s` = `"mb"`, `"be"` או `"fd"` |
| `bose_einstein`, `fermi_dirac`, `maxwell_boltzmann`, `occupation(eps, mu, T, s)` | האכלוס הממוצע של מצב בעל אנרגיה `eps` |
| `occupation_variance`, `explicit_occupation`, `sample_occupation` | השונות שלו; ממוצע ושונות מתוך הסכום הגרנד-קנוני המפורש; הגרלות אקראיות מתוך ההתפלגות שלו |
| `box_level_unit`, `count_box_states`, `smooth_state_count`, `density_of_states_3d` | המצבים של קובייה, אחד אחד ובספירה חלקה; $g(\varepsilon)$ |
| `degeneracy_parameter(n, T, m)`, `occupation_integral(eta, s)`, `solve_eta`, `solve_mu` | $n\lambda^3$; $F(\eta)$; ה-$\mu$ ששומר על הצפיפות |
| `ideal_quantum_gas(n, T, m, s, g_s)` | $\mu$, אנרגיה, לחץ, קיבול חום וחלק הקונדנסט של גז |
| `bec_tc`, `condensate_fraction`, `finite_box_condensate_fraction` | עיבוי, אידיאלי ובקופסה של $N$ אטומים |
| `fermi_energy`, `fermi_temperature`, `fermi_velocity`, `fermi_gas_pressure`, `electronic_heat_capacity`, `free_electron_gamma` | ים פרמי |
| `fit_metal_heat_capacity` | הקו הישר $C/T = \gamma + \beta T^2$, ומחזירה `.gamma`, `.gamma_sigma`, `.beta`, `.debye_temperature` |

### לנבא

התחייבו לתשובה לכל אחת מהשאלות לפני שתריצו משהו.

1. שני בוזונים זהים בשני מצבים: מה הסיכוי שהם נמצאים באותו מצב?
2. כאשר גז מקורר בצפיפות קבועה, לאיזה כיוון נע $\mu$ — עבור בוזונים, עבור פרמיונים?
3. במחצית טמפרטורת העיבוי, איזה חלק מגז בוזה אידיאלי נמצא במצב הנמוך ביותר?
4. המקדם האלקטרוני $\gamma$ של כסף: עד כדי איזה גורם הוא מסכים עם ניבוי האלקטרונים החופשיים?

**הניבויים שלכם:**

1.
2.
3.
4.

## חלק 2 — ספירה

רשמו את המצבים של שני חלקיקים בשני מצבים חד-חלקיקיים תחת כל כלל, ואת הסיכוי למצוא את שניהם באותו
מצב אם כל המצבים שווי הסתברות. אחר כך השוו את הספירות הקוונטיות של שלושה חלקיקים ב-$M$ מצבים לספירה
הקלאסית המתוקנת $M^3/3!$ של מודול 12.

**הניבוי שלכם:** עבור שלושה חלקיקים ב-$1000$ מצבים, עד כמה קרובות הספירות של בוזה ושל פרמי
ל-$M^3/3!$?

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
together = {}
for statistics, name in (("mb", "classical (labelled)"), ("be", "bosons"), ("fd", "fermions")):
    states = quantum.enumerate_states(2, 2, statistics)
    together[statistics] = float(np.mean(states.max(axis=1) == 2))
    print(f"{name:21s} {len(states)} states {[tuple(int(k) for k in s) for s in states]}, "
          f"P(both in one state) = {together[statistics]:.3f}")

print("\nthree particles in M states, divided by the corrected classical count M^3 / 3!:")
for levels in (3, 10, 100, 1000):
    classical = levels**3 / 6
    print(f"M = {levels:5d}: bosons {quantum.count_states(levels, 3, 'be') / classical:.4f}, "
          f"fermions {quantum.count_states(levels, 3, 'fd') / classical:.4f}")

הספירות $4$, $3$, $1$ הן כל הקלט הקוונטי. בוזונים נמצאים יחד לעתים קרובות יותר מחלקיקים קלאסיים,
ופרמיונים לעולם לא, בלי שום כוח בשום מקום. וכאשר מספר המצבים עולה בהרבה על מספר החלקיקים, שתי
הספירות הקוונטיות הופכות ל-$M^N/N!$: החלוקה ב-$N!$ של מודול 12 הייתה סטטיסטיקה קוונטית בגבול המדולל
שלה.

## חלק 3 — מצב אחד, והפלוקטואציות שלו

כל ביטוי סגור של שלב 2 נבדק מול הסכום המפורש של המשקלים הגרנד-קנוניים של המצב —
$n = 0, 1, 2, \ldots$ עבור בוזונים, $0$ או $1$ עבור פרמיונים, ועם $1/n!$ עבור חלקיקים קלאסיים. אחר
כך מוגרלים אכלוסים באקראי מכל התפלגות, בעזרת מחולל עם זרע.

**הניבוי שלכם:** לאיזו סטטיסטיקה השונות הגדולה ביותר באותו אכלוס ממוצע?

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
T = 300.0
print("   x     statistics   closed form      explicit sum     variance (closed)  variance (sum)")
worst = 0.0
for x in (0.1, 1.0, 3.0):
    for statistics in ("be", "mb", "fd"):
        eps = x * K_B * T
        mean, variance = quantum.explicit_occupation(eps, 0.0, T, statistics)
        closed = quantum.occupation(eps, 0.0, T, statistics)
        closed_variance = quantum.occupation_variance(eps, 0.0, T, statistics)
        worst = max(worst, relative_error(mean, closed), relative_error(variance, closed_variance))
        print(f"{x:5.1f}   {statistics:>6s}      {closed:.12f}   {mean:.12f}   "
              f"{closed_variance:.12f}   {variance:.12f}")
print(f"largest relative difference: {worst:.1e}")

rng = np.random.default_rng(17)
eps = 0.5 * K_B * T
print("\n100 000 draws of one state's occupation at (eps - mu) / kT = 0.5:")
for statistics in ("be", "mb", "fd"):
    draws = quantum.sample_occupation(eps, 0.0, T, statistics, 100_000, rng)
    print(f"  {statistics}: mean {draws.mean():.4f} (predicted "
          f"{quantum.occupation(eps, 0.0, T, statistics):.4f}), variance {draws.var():.4f} "
          f"(predicted {quantum.occupation_variance(eps, 0.0, T, statistics):.4f})")

x = np.linspace(0.02, 6.0, 300)
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
for statistics, colour, label in (("be", "#2563eb", "Bose-Einstein"),
                                  ("mb", "#111827", "Maxwell-Boltzmann"),
                                  ("fd", "#dc2626", "Fermi-Dirac")):
    n = quantum.occupation(x * K_B * T, 0.0, T, statistics)
    left.plot(x, n, color=colour, label=label)
    right.semilogy(x, n, color=colour, label=label)
left.set_ylim(0, 3)
for ax in (left, right):
    ax.set_xlabel("(eps - mu) / k_B T")
    ax.set_ylabel("<n>")
    ax.legend()
plt.tight_layout()
plt.show()

הביטויים הסגורים הם הסכומים, עד כדי עיגול. השונויות, $n(1 + n)$, $n$ ו-$n(1 - n)$, מסדרות את שלוש
הסטטיסטיקות באותו סדר שבו סידרה אותן הספירה: בוזונים מתקבצים, פרמיונים נמנעים זה מזה. בגרף הלוגריתמי
שלושת האכלוסים הופכים לקווים מקבילים ברגע ש-$(\varepsilon - \mu)/\kB T$ מגיע לכמה יחידות: הגבול
המדולל.

## חלק 4 — המצבים של קופסה

לגל העומד של חלקיק בקובייה שאורך צלעה $L$ יש $\mathbf{k} = (\pi/L)(n_x, n_y, n_z)$ ואנרגיה
$\varepsilon_1 |\mathbf{n}|^2$, כאשר $\varepsilon_1 = \hbar^2\pi^2/2mL^2$. `count_box_states` סופרת
את שלשות המספרים השלמים אחת אחת; הגזירה החליפה אותן בנפח של אוקטנט.

In [ ]:
side = 1e-6  # m
unit = quantum.box_level_unit(side, M_HE)
shortfalls = {}
print(" radius |n|   explicit count   smooth count   shortfall   shortfall x |n|")
for radius in (5, 10, 20, 40, 80, 160, 320):
    counted = quantum.count_box_states(side, M_HE, unit * radius**2)
    smooth = quantum.smooth_state_count(side, M_HE, unit * radius**2)
    shortfalls[radius] = 1 - counted / smooth
    print(f"{radius:10d} {counted:16d} {smooth:14.0f}   {100 * shortfalls[radius]:7.3f} %   "
          f"{shortfalls[radius] * radius:6.3f}")
print(f"\nhelium in a 1 cm box: eps_1 / k_B = {quantum.box_level_unit(0.01, M_HE) / K_B:.1e} K")

מכפלת החוסר ב-$|\mathbf{n}|$ מתייצבת על $9/4$: הספירה החלקה סופרת ביתר רק את המצבים שעל פאות
האוקטנט, בדיוק כמו בחלל של מודול 16. בכל קופסה שתוכלו לבנות, המרווח בין הרמות הוא כמה פעמים
$10^{-15}\ \mathrm{K}$, וצפיפות המצבים החלקה מדויקת לכל צורך מעשי.

## חלק 5 — הפוטנציאל הכימי בצפיפות קבועה

בצפיפות של הליום נוזלי, מצאו את $\mu$ בכל טמפרטורה עבור בוזונים, עבור חלקיקים קלאסיים ועבור פרמיונים
(היפותטיים, חסרי ספין), ובדקו את הפותר הקלאסי מול הביטוי הסגור של מודול 13, $\mu = \kB T\ln(n/n_Q)$.

**הניבוי שלכם:** שרטטו את $\mu/\kB T_c$ כנגד $T/T_c$ עבור שלושת הגזים לפני ההרצה.

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
ratios = np.geomspace(0.3, 30.0, 80)  # T / T_c
mu_curves = {s: np.array([quantum.solve_mu(N_HE, r * T_C, M_HE, s) for r in ratios]) / (K_B * T_C)
             for s in ("mb", "be", "fd")}
module_13 = np.array([chemical.ideal_gas_mu(N_HE, r * T_C, M_HE) for r in ratios]) / (K_B * T_C)
mb_gap = float(np.max(np.abs(mu_curves["mb"] - module_13) / ratios))
print(f"classical solver against module 13's k_B T ln(n / n_Q): largest difference "
      f"{mb_gap:.1e} k_B T")
eps_f_spinless = quantum.fermi_energy(N_HE, M_HE, 1) / (K_B * T_C)
print(f"spinless fermions at this density: eps_F = {eps_f_spinless:.3f} k_B T_c")
print(f"the classical mu crosses zero at T / T_c = {quantum.ZETA_3_2 ** (2 / 3):.4f}")

print("\nn lambda^3   mu/kT: classical   bosons   fermions")
for d in (0.01, 0.1, 1.0, quantum.ZETA_3_2, 10.0):
    print(f"{d:10.3f}   {quantum.solve_eta(d, 'mb'):+16.3f}   {quantum.solve_eta(d, 'be'):+6.3f}   "
          f"{quantum.solve_eta(d, 'fd'):+8.3f}")

fig, ax = plt.subplots(figsize=(8, 4.5))
for statistics, colour, label in (("be", "#2563eb", "bosons"), ("mb", "#111827", "classical"),
                                  ("fd", "#dc2626", "fermions (spinless)")):
    ax.plot(ratios, mu_curves[statistics], color=colour, label=label)
ax.axhline(eps_f_spinless, color="#dc2626", ls=":", lw=1)
ax.axhline(0.0, color="grey", lw=1)
ax.axvline(1.0, color="#2563eb", ls=":", lw=1)
ax.set_xscale("log")
ax.set_ylim(-20, 4)
ax.set_xlabel("T / T_c")
ax.set_ylabel("mu / k_B T_c")
ax.legend()
plt.tight_layout()
plt.show()

שלושתם מטפסים כשהגז מתקרר. ה-$\mu$ הקלאסי חוצה את האפס ב-$1.90\,T_c$, שם הוא מתחיל לנבא יותר מחלקיק
אחד במצבים הנמוכים ביותר — התאוריה הקלאסית יוצאת מתחומה. ה-$\mu$ של הבוזונים מגיע לאפס ב-$T_c$ ונשאר
שם. זה של הפרמיונים מטפס עד $\varepsilon_F$.

## חלק 6 — עיבוי בוזה–איינשטיין

ראשית, הקיבולת של המצבים המעוררים: אינטגרל האכלוס ב-$\mu = 0$, באינטגרציה נומרית. אחר כך חלק
הקונדנסט — האידיאלי, וזה המחושב רמה אחר רמה עבור קופסה בעלת דפנות קשיחות של $N$ אטומים. אחר כך הלחץ
מתחת ל-$T_c$.

**הניבוי שלכם:** בקופסה של $10^4$ אטומים, האם $N_0/N$ ב-$T_c/2$ גבוה מהערך האידיאלי
$1 - 0.5^{3/2} = 0.646$ או נמוך ממנו?

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
capacity = quantum.occupation_integral(0.0, "be")
print(f"F_BE(0) by quadrature: {capacity:.16f};  zeta(3/2) = {quantum.ZETA_3_2:.16f}")
for n_points in (16, 24, 32, 64):
    error = relative_error(quantum.occupation_integral(0.0, "be", 1.5, n_points), quantum.ZETA_3_2)
    print(f"  {n_points:3d} trapezoid intervals: relative error {error:.1e}")

t = np.linspace(0.0, 1.5, 151)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(t, quantum.condensate_fraction(t, 1.0), color="k", lw=2, label="ideal, 1 - (T/T_c)^(3/2)")
for n_atoms, colour in ((1000, "#94a3b8"), (10_000, "#2563eb"), (100_000, "#d97706")):
    ax.plot(t, quantum.finite_box_condensate_fraction(n_atoms, t), ls="--", color=colour,
            label=f"box of {n_atoms:,} atoms")
ax.set_xlabel("T / T_c")
ax.set_ylabel("N_0 / N")
ax.legend()
plt.tight_layout()
plt.show()

window = np.linspace(0.2, 0.8, 7)
exponents, excess = {}, {}
print("\n    atoms   N0/N at T_c/2   excess over 0.646   fitted exponent of 1 - N0/N")
for n_atoms in (1000, 10_000, 100_000, 1_000_000):
    half = float(quantum.finite_box_condensate_fraction(n_atoms, 0.5))
    excess[n_atoms] = half - (1 - 0.5**1.5)
    fraction = np.asarray(quantum.finite_box_condensate_fraction(n_atoms, window))
    exponents[n_atoms] = scaling_exponent(window, 1 - fraction)
    print(f"{n_atoms:9d}   {half:13.4f}   {excess[n_atoms]:17.4f}   {exponents[n_atoms]:12.3f}")

print("\npressure at T = T_c / 2, for three densities:")
pressures = []
for factor in (1, 2, 5):
    gas = quantum.ideal_quantum_gas(factor * N_HE, 0.5 * T_C, M_HE, "be")
    pressures.append(gas.pressure)
    print(f"  {factor} x n: N0/N = {gas.condensate_fraction:.3f}, P = {gas.pressure:.5e} Pa")
at_tc = quantum.ideal_quantum_gas(N_HE, T_C, M_HE, "be")
print(f"at T_c: P / (n k_B T) = {at_tc.pressure / at_tc.classical_pressure:.4f}, "
      f"C / (N k_B) = {at_tc.heat_capacity_per_particle / K_B:.4f}")

האינטגרציה הנומרית מוצאת את $\zeta(3/2)$ בלי שאומרים לה אותו, עם $32$ מרווחים. קופסה סופית מתעבה
*מוקדם*: הדפנות הקשיחות שלה מסלקות את המצבים שעל פאות האוקטנט, ולכן המצבים המעוררים מתמלאים מוקדם
יותר. העודף מצטמק לאט ככל שהקופסה גדלה, והמעריך המותאם נסחף כלפי מטה לעבר $3/2$. מתחת ל-$T_c$ הלחץ
זהה בכל צפיפות: האטומים הנוספים מצטרפים לקונדנסט, שאינו לוחץ על דבר, בעוד האטומים המעוררים ממשיכים
לנוע.

## חלק 7 — ים פרמי

אלקטרונים בצפיפות של נחושת, עם $g_s = 2$: אנרגיית פרמי ולחץ הניוון בביטוי סגור; אחר כך הגז בפתרון
נומרי, בטמפרטורות מ-$0.001\,T_F$ עד $3\,T_F$.

**הניבוי שלכם:** ב-$T = 0.01\,T_F$, איזה חלק מהערך הקלאסי $\tfrac32\kB$ לאלקטרון הוא קיבול החום?

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
eps_f = float(quantum.fermi_energy(N_CU, M_E))
print(f"copper: eps_F = {eps_f / EV:.3f} eV, T_F = {T_F:.0f} K, "
      f"v_F = {quantum.fermi_velocity(N_CU, M_E):.3e} m/s")
print(f"degeneracy pressure (2/5) n eps_F = {quantum.fermi_gas_pressure(N_CU, M_E):.3e} Pa; "
      f"classical n k_B T at 300 K = {N_CU * K_B * 300:.3e} Pa")
print(f"n lambda^3 at 300 K = {quantum.degeneracy_parameter(N_CU, 300.0, M_E):.0f}; "
      f"T / T_F = {300 / T_F:.4f}")

cold = quantum.ideal_quantum_gas(N_CU, 1e-3 * T_F, M_E, "fd", 2)
print(f"\nat T = 0.001 T_F: mu / eps_F = {cold.mu / eps_f:.8f}, "
      f"U / (N eps_F) = {cold.energy_per_particle / eps_f:.8f} (3/5 = 0.6)")

fermi_ratios = np.geomspace(1e-3, 3.0, 60)
fermi_gases = [quantum.ideal_quantum_gas(N_CU, r * T_F, M_E, "fd", 2) for r in fermi_ratios]
heat = np.array([g.heat_capacity_per_particle for g in fermi_gases]) / K_B
sommerfeld_ratio = heat[0] / (np.pi**2 / 2 * fermi_ratios[0])
print(f"C / (N k_B) at T = 0.001 T_F is {sommerfeld_ratio:.6f} times (pi^2/2) T/T_F")

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
left.semilogx(fermi_ratios, [g.mu / eps_f for g in fermi_gases], color="#dc2626")
left.axhline(1.0, color="grey", ls=":")
left.set_xlabel("T / T_F")
left.set_ylabel("mu / eps_F")
right.loglog(fermi_ratios, heat, color="#2563eb", label="ideal Fermi gas")
right.loglog(fermi_ratios, np.pi**2 / 2 * fermi_ratios, "--", color="#dc2626",
             label="(pi^2/2) T/T_F")
right.axhline(1.5, color="k", ls=":", label="classical 3/2")
right.set_ylim(1e-3, 5)
right.set_xlabel("T / T_F")
right.set_ylabel("C_V / (N k_B)")
right.legend()
plt.tight_layout()
plt.show()

באפס המוחלט האלקטרונים שומרים על אנרגיה ממוצעת של $\tfrac35\varepsilon_F$, יותר מארבעה אלקטרון-וולט,
ועל לחץ גדול פי מאה מהלחץ הקלאסי בטמפרטורת החדר. קיבול החום לינארי ב-$T$ עם המקדם $\pi^2/2$ עד
כ-$0.05\,T_F$ — עבור נחושת, $4000\ \mathrm{K}$, מעל נקודת ההתכה שלה.

## חלק 8 — חוק פלאנק כגז בוזה

מודול 16 בנה את הספקטרום של פלאנק מהמתנד. כאן בנו אותו מחדש מהאכלוס של מודול 17: פוטונים הם בוזונים
עם $\mu = 0$, ולכן אופן בתדירות $\nu$ מחזיק `bose_einstein(h nu, 0, T)` פוטונים בעלי אנרגיה $h\nu$,
ויש `radiation.mode_density(nu)` אופנים ליחידת נפח לכל הרץ.

In [ ]:
planck_gap = 0.0
for temperature in (2.725, 300.0, 5772.0):
    nu = np.linspace(1e-3, 30.0, 3000) * K_B * temperature / H
    rebuilt = radiation.mode_density(nu) * H * nu * quantum.bose_einstein(H * nu, 0.0, temperature)
    gap = float(np.max(np.abs(rebuilt / radiation.planck_u_nu(nu, temperature) - 1.0)))
    planck_gap = max(planck_gap, gap)
    print(f"T = {temperature:7.3f} K: largest relative difference from planck_u_nu {gap:.1e}")

nu = np.linspace(1e12, 1.5e15, 600)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(nu / 1e12, radiation.planck_u_nu(nu, 5772.0), color="#2563eb", lw=4, alpha=0.4,
        label="module 16: planck_u_nu")
ax.plot(nu / 1e12, radiation.mode_density(nu) * H * nu * quantum.bose_einstein(H * nu, 0.0, 5772.0),
        color="k", lw=1, label="module 17: modes x h nu x n_BE(mu = 0)")
ax.set_xlabel("frequency (THz)")
ax.set_ylabel("u_nu (J s m^-3)")
ax.legend()
plt.tight_layout()
plt.show()

פונקציית אכלוס אחת, שני פרקים: שני החישובים מסכימים עד הספרה האחרונה של מספר בדיוק כפול.

## חלק 9 — האלקטרונים בכסף

קובץ הנתונים הוא זה שבו השתמש מודול 16: קיבול החום של כסף בלחץ קבוע מ-$1$ עד $300\ \mathrm{K}$, כפי
שרוכז בטבלאות של NIST (סמית' ופיקט, 1995), לקילוגרם. מתחת לקלווינים ספורים
$C = \gamma T + \beta T^3$; הפעם ה-$\gamma$ של האלקטרונים הוא המדידה, וגז האלקטרונים החופשיים מנבא
אותו מתוך הצפיפות של כסף בלבד.

**הניבוי שלכם:** מתחת לאיזו טמפרטורה נושאים האלקטרונים של כסף חלק גדול יותר מקיבול החום שלו מזה
שנושא הסריג שלו?

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
MOLAR_MASS, DENSITY = 0.1078682, 10490.0  # kg/mol, kg/m^3
t_ag, cp_per_kg = np.genfromtxt(DATA / "16-silver-heat-capacity.csv", delimiter=",",
                                comments="#").T
c_ag = cp_per_kg * MOLAR_MASS  # J/(mol K)

fits = {}
for t_max in (4.0, 6.0, 8.0):
    window = t_ag <= t_max
    fits[t_max] = quantum.fit_metal_heat_capacity(t_ag[window], c_ag[window])
    print(f"T <= {t_max:.0f} K: gamma = {fits[t_max].gamma * 1e3:.4f} +- "
          f"{fits[t_max].gamma_sigma * 1e3:.4f} mJ/(mol K^2), "
          f"theta_D = {fits[t_max].debye_temperature:.1f} K")
fit = fits[6.0]

n_ag = DENSITY / MOLAR_MASS * N_A
gamma_free = quantum.free_electron_gamma(n_ag)
print(f"\none electron per atom: n = {n_ag:.4e} m^-3, "
      f"eps_F = {quantum.fermi_energy(n_ag, M_E) / EV:.3f} eV, "
      f"T_F = {quantum.fermi_temperature(n_ag, M_E):.0f} K")
print(f"free-electron gamma = {gamma_free * 1e3:.4f} mJ/(mol K^2); measured / free = "
      f"{fit.gamma / gamma_free:.3f}")
crossover = float(np.sqrt(fit.gamma / fit.beta))
share = fit.gamma * t_ag / c_ag
print(f"electrons and lattice are equal at {crossover:.2f} K; electrons' share at 1 K "
      f"{share[0]:.2f}, at 300 K {share[-1]:.4f}")

# Measured gammas from Kittel's table (8th ed., p. 139), against the free-electron values
# computed from each metal's room-temperature density and molar mass.
KITTEL = {"Ag": (10490.0, 107.8682e-3, 1, 0.646e-3), "Au": (19300.0, 196.96657e-3, 1, 0.729e-3),
          "Cu": (8960.0, 63.546e-3, 1, 0.695e-3), "Al": (2700.0, 26.9815385e-3, 3, 1.35e-3)}
print("\nmetal   measured   free electron   ratio")
for metal, (rho, molar, z, measured) in KITTEL.items():
    free = quantum.free_electron_gamma(z * rho / molar * N_A, z)
    print(f"{metal:5s}   {measured * 1e3:8.3f}   {free * 1e3:13.3f}   {measured / free:5.2f}")

low = t_ag <= 6.0
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
grid = np.linspace(0, 37, 50)
left.plot(t_ag[low] ** 2, c_ag[low] / t_ag[low] * 1e3, "o", color="k", label="silver")
left.plot(grid, (fit.gamma + fit.beta * grid) * 1e3, color="#2563eb", label="fit")
left.plot([0], [gamma_free * 1e3], "D", color="#dc2626", label="free-electron gamma")
left.set_xlabel("T^2 (K^2)")
left.set_ylabel("C / T (mJ mol^-1 K^-2)")
left.legend()
right.semilogx(t_ag, share, "o", ms=3, color="k")
right.set_xlabel("T (K)")
right.set_ylabel("gamma T / C")
plt.tight_layout()
plt.show()

גז האלקטרונים החופשיים, בלי שום פרמטר מתכוונן, מנבא את ה-$\gamma$ של כסף בדיוק של כ-$1\%$.
האלקטרונים נושאים את רוב קיבול החום של כסף מתחת ל-$2\ \mathrm{K}$, ופחות מ-$1\%$ ממנו בטמפרטורת
החדר: קיבול החום "החסר" של החידה, נמדד. מתכות אחרות חופשיות פחות — היחס של נחושת הוא $1.38$ — והיחס
מודד כיצד הסריג מעצב מחדש את המצבים בראש ים פרמי.

## חלק 10 — הליום נוזלי

גז בוזה האידיאלי בצפיפות של הליום נוזלי מתעבה ב-$T_c$. הנוזל הופך לנוזל-על בנקודת למדא,
$2.1768\ \mathrm{K}$ (דונלי וברנגי, 1998).

In [ ]:
T_LAMBDA = 2.1768  # K, measured
print(f"ideal-gas T_c = {T_C:.3f} K; lambda point {T_LAMBDA} K; ratio {T_C / T_LAMBDA:.3f}")
print(f"spacing n^(-1/3) = {N_HE ** (-1 / 3) * 1e9:.3f} nm; thermal wavelength at the lambda point "
      f"= {thermal_wavelength(T_LAMBDA, M_HE) * 1e9:.3f} nm")
print(f"n lambda^3 at the lambda point = {quantum.degeneracy_parameter(N_HE, T_LAMBDA, M_HE):.2f}; "
      f"ideal condensate fraction there = {quantum.condensate_fraction(T_LAMBDA, T_C):.3f}")

הגז האידיאלי מחטיא את נקודת למדא ב-$45\%$ — סדר הגודל הנכון מתוך המסה והצפיפות בלבד, עבור נוזל
שאטומיו יושבים זה ליד זה במרחק קטן מאורך הגל התרמי שלהם, ודוחים זה את זה בחוזקה. המודל מאתר את
התופעה; הוא אינו מתאר את הנוזל.

### מדידה

צטטו שתי תוצאות, כל אחת עם אי-הוודאות שלה ועם ההנחות שהיא נשענת עליהן:

- ה-$\gamma$ של כסף מחלק 9, מול ניבוי האלקטרונים החופשיים — וציינו כמה הוא זז אם החלון מסתיים
  ב-$4\ \mathrm{K}$ במקום ב-$6$.
- המעריך של $1 - N_0/N$ עבור קופסה של $10^6$ אטומים מחלק 6, מול הערך האידיאלי $3/2$ — וציינו מה
  מרמזת המגמה כפונקציה של $N$.

## חלק 11 — הבדיקות האוטומטיות

In [ ]:
# 1. The closed forms are the explicit grand sums.
assert worst < 1e-12

# 2. Counting: 4, 3, 1 states; bosons bunch, fermions avoid.
assert [quantum.count_states(2, 2, s) for s in ("mb", "be", "fd")] == [4, 3, 1]
assert together["be"] > together["mb"] > together["fd"] == 0.0

# 3. The smooth density of states misses only the octant's faces.
assert 0 < shortfalls[320] < shortfalls[80] < shortfalls[20]
assert abs(shortfalls[320] * 320 - 9 / 4) < 0.1

# 4. The classical solver is module 13's formula.
assert mb_gap < 1e-12

# 5. Condensation: zeta(3/2) by quadrature; finite boxes approach the ideal from above.
assert relative_error(capacity, quantum.ZETA_3_2) < 1e-14
assert all(e > 0 for e in excess.values())
assert exponents[1000] > exponents[1_000_000] > 1.5 and exponents[1_000_000] < 1.6
assert max(pressures) / min(pressures) - 1 < 1e-12

# 6. The Fermi sea: mu -> eps_F, U/N -> (3/5) eps_F, C linear with pi^2/2.
assert abs(cold.mu / eps_f - 1) < 1e-5 and abs(cold.energy_per_particle / eps_f - 0.6) < 1e-5
assert abs(sommerfeld_ratio - 1) < 1e-4

# 7. Planck's law is the Bose-Einstein distribution at mu = 0.
assert planck_gap < 1e-12

# 8. Silver's electrons are nearly free; the ideal Bose gas misses helium's lambda point.
assert 0.6e-3 < fit.gamma < 0.7e-3 and abs(fit.gamma / gamma_free - 1) < 0.03
assert 1.5 < crossover < 2.5
assert 3.1 < T_C < 3.2
print("all checks passed")

## חלק 12 — חקרו בעצמכם

ארבעה כלי חקירה; כוונו את הפקדים ולחצו על **Run Interact**.

**כלי החקירה של האכלוס.** שלושת האכלוסים כנגד האנרגיה, עבור הטמפרטורה והפוטנציאל הכימי שתבחרו (שניהם
באותה יחידת אנרגיה שרירותית). דברים לנסות: $\mu$ הרבה מתחת לאפס, בציר הלוגריתמי; אחר כך העלו את
$\mu$ לעבר האפס; אחר כך נסו לדחוף את ה-$\mu$ של הבוזונים אל מעל לאפס; ואחר כך הורידו את הטמפרטורה
כאשר $\mu$ חיובי וצפו בפונקציית פרמי מתחדדת.

In [ ]:
import ipywidgets as widgets


def occupations(temperature=1.0, mu=-1.0, log_axis=False):
    eps = np.linspace(0.0, 8.0, 801)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    # temperature / K_B makes k_B T equal to `temperature`, in the same unit as eps and mu.
    for statistics, colour, label in (("be", "#2563eb", "Bose-Einstein"),
                                      ("mb", "#111827", "Maxwell-Boltzmann"),
                                      ("fd", "#dc2626", "Fermi-Dirac")):
        if statistics == "be" and mu >= 0.0:
            print("No Bose-Einstein curve: mu must lie below the lowest energy, 0, or the lowest "
                  "state's grand sum diverges.")
            continue
        ax.plot(eps, quantum.occupation(eps, mu, temperature / K_B, statistics), color=colour,
                label=label)
    ax.axvline(mu, color="grey", ls=":", lw=1)
    if log_axis:
        ax.set_yscale("log")
        ax.set_ylim(1e-4, 50)
    else:
        ax.set_ylim(0, 3)
    ax.set_xlabel("eps (energy unit)")
    ax.set_ylabel("<n>")
    ax.legend()
    plt.tight_layout()
    plt.show()
    for name, n, t, m in (("helium gas, 0 C, 1 atm", 101325 / (K_B * 273.15), 273.15, M_HE),
                          ("liquid helium, 2.2 K", N_HE, 2.2, M_HE),
                          ("electrons in copper, 300 K", N_CU, 300.0, M_E)):
        print(f"n lambda^3, {name}: {quantum.degeneracy_parameter(n, t, m):.2e}")


widgets.interact_manual(
    occupations,
    temperature=widgets.FloatLogSlider(value=1.0, base=10, min=-1.5, max=1.0, step=0.05),
    mu=widgets.FloatSlider(value=-1.0, min=-6.0, max=4.0, step=0.1),
    log_axis=widgets.Checkbox(value=False, description="log axis"),
)

**כלי החקירה של העיבוי.** חלק הקונדנסט של קופסה של $N$ אטומים, הנספר רמה אחר רמה, מול העקומה
האידיאלית, עם החלוקה בין מצב היסוד למצבים המעוררים בטמפרטורה שתבחרו. דברים לנסות: $100$ אטומים, ואז
$100\,000$; $T = T_c$ לכל אחד מהם.

In [ ]:
def condensation(n_atoms=10_000, t_over_tc=0.5):
    t = np.linspace(0.0, 1.5, 76)
    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [2.5, 1]})
    left.plot(t, quantum.condensate_fraction(t, 1.0), color="k", lw=2, label="ideal")
    left.plot(t, quantum.finite_box_condensate_fraction(n_atoms, t), "--", color="#2563eb",
              label=f"box of {n_atoms:,} atoms")
    here = float(quantum.finite_box_condensate_fraction(n_atoms, t_over_tc))
    left.plot([t_over_tc], [here], "o", color="#d97706", ms=9)
    left.set_xlabel("T / T_c")
    left.set_ylabel("N_0 / N")
    left.legend()
    right.bar(["ground", "excited"], [here, 1 - here], color=["#1e3a8a", "#93c5fd"])
    right.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()
    print(f"N0/N = {here:.4f} in the box; ideal {quantum.condensate_fraction(t_over_tc, 1.0):.4f}")


widgets.interact_manual(
    condensation,
    n_atoms=widgets.SelectionSlider(options=[100, 1000, 10_000, 100_000], value=10_000),
    t_over_tc=widgets.FloatSlider(value=0.5, min=0.0, max=1.5, step=0.01),
)

**כלי החקירה של ים פרמי.** החלק המאוכלס של צפיפות המצבים, $g(\varepsilon)\,n_{\mathrm{FD}}$, מתחת
ל-$g(\varepsilon)$, עבור אלקטרונים בצפיפות של נחושת, עם הרצועה $\mu \pm \kB T$ שעדיין יכולה לקלוט
חום. דברים לנסות: $T/T_F = 1$, $0.1$ ו-$0.01$; קראו בכל פעם את קיבול החום.

In [ ]:
def fermi_sea(t_over_tf=0.1):
    eps = np.linspace(0.0, 2.0, 600)  # in units of eps_F
    gas = quantum.ideal_quantum_gas(N_CU, t_over_tf * T_F, M_E, "fd", 2)
    mu = gas.mu / (K_B * T_F)
    density = 1.5 * np.sqrt(eps)  # g(eps) eps_F / N
    filled = density * quantum.occupation(eps, mu, t_over_tf / K_B, "fd")
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(eps, density, color="grey")
    ax.fill_between(eps, filled, color="#93c5fd")
    ax.axvspan(max(mu - t_over_tf, 0.0), mu + t_over_tf, color="#d97706", alpha=0.25)
    ax.axvline(1.0, color="k", ls="--", lw=1)
    ax.set_xlabel("eps / eps_F")
    ax.set_ylabel("g(eps) eps_F / N")
    plt.tight_layout()
    plt.show()
    print(f"mu / eps_F = {mu:.4f};  C / (N k_B) = {gas.heat_capacity_per_particle / K_B:.4f} "
          f"(Sommerfeld (pi^2/2) T/T_F = {np.pi**2 / 2 * t_over_tf:.4f}; classical 1.5)")


widgets.interact_manual(
    fermi_sea,
    t_over_tf=widgets.FloatLogSlider(value=0.1, base=10, min=-2, max=0.5, step=0.05),
)

**כלי המעקב אחר הפוטנציאל הכימי.** שלוש העקומות של חלק 5, ועליהן מסומנים שלושת הגזים בטמפרטורה
שתבחרו. דברים לנסות: מעט מעל $T_c$ ומעט מתחתיה; $T = 1.9\,T_c$, שם ה-$\mu$ הקלאסי הוא אפס;
$T = 30\,T_c$, שם שלושתם מסכימים.

In [ ]:
def mu_tracker(t_over_tc=1.5):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for statistics, colour in (("be", "#2563eb"), ("mb", "#111827"), ("fd", "#dc2626")):
        ax.plot(ratios, mu_curves[statistics], color=colour)
        here = quantum.solve_mu(N_HE, t_over_tc * T_C, M_HE, statistics) / (K_B * T_C)
        ax.plot([t_over_tc], [here], "o", color=colour, ms=8)
        print(f"{statistics}: mu = {here:+.4f} k_B T_c")
    ax.axhline(0.0, color="grey", lw=1)
    ax.set_xscale("log")
    ax.set_ylim(-20, 4)
    ax.set_xlabel("T / T_c")
    ax.set_ylabel("mu / k_B T_c")
    plt.tight_layout()
    plt.show()
    print(f"n lambda^3 = {quantum.degeneracy_parameter(N_HE, t_over_tc * T_C, M_HE):.4f}")


widgets.interact_manual(
    mu_tracker,
    t_over_tc=widgets.FloatLogSlider(value=1.5, base=10, min=-0.5, max=1.45, step=0.01),
)

## בחנו את הבנתכם

הריצו את התא שלהלן לחידון בבדיקה אוטומטית. אותן שאלות, עם הסברים כתובים לכל אפשרות, נמצאות בעמוד
המודול.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "17-quantum-gases.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שתלכו

כתבו כמה משפטים על כל אחת, בתא שלהלן.

1. מה ניבאתם שהתברר כשגוי, ומה בדיוק היה הפגם בהנמקה שלכם?
2. חלק 2 מצא בוזונים יחד לעתים קרובות יותר מחלקיקים קלאסיים, בלי שום כוח ביניהם. הסבירו מאין מגיעה
   ההסתברות הנוספת.
3. המדידה שלכם של ה-$\gamma$ של כסף מצטטת אי-ודאות. מנו כל הנחה שהמדידה נשענת עליה ושאי-הוודאות
   *אינה* כוללת.
4. מתחת ל-$T_c$ הלחץ של גז בוזה אינו תלוי בצפיפות שלו. הסבירו מדוע, במונחים של מה שהאטומים המעובים
   עושים.

**התשובות שלכם:**

1.
2.
3.
4.